# Entrenamiento CNN — Detección de Señales de Tránsito (GTSRB)

**Proyecto Final IA — FUP**  
Dataset: GTSRB — 43 clases, ~50.000 imágenes  
Hardware recomendado: GPU T4 (Kaggle Notebooks — gratuito)

## Instrucciones antes de ejecutar
1. En Kaggle: Settings → Accelerator → **GPU T4**
2. Agregar dataset: `meowmeowmeowmeowmeow/gtsrb-german-traffic-sign`
3. Ejecutar todas las celdas en orden

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

GTSRB_DIR = Path('/kaggle/input/gtsrb-german-traffic-sign')
TRAIN_DIR = GTSRB_DIR / 'Train'
TEST_CSV  = GTSRB_DIR / 'Test.csv'
OUT_DIR   = Path('/kaggle/working')

IMG_SIZE      = 32
NUM_CLASSES   = 43
BATCH_SIZE    = 64
EPOCHS        = 30
LEARNING_RATE = 1e-3
VAL_SPLIT     = 0.2

In [ ]:
# Carga del dataset de entrenamiento
print('Cargando imágenes de entrenamiento...')
images, labels = [], []

for cls_dir in sorted(TRAIN_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    cls_id = int(cls_dir.name)
    for img_path in cls_dir.glob('*.png'):
        img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        images.append(np.array(img, dtype=np.float32) / 255.0)
        labels.append(cls_id)

X = np.array(images)
y = np.array(labels, dtype=np.int32)
print(f'Total imágenes: {len(X)} | Clases únicas: {len(np.unique(y))}')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=VAL_SPLIT, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Val: {len(X_val)}')

In [ ]:
# Carga del conjunto de test
print('Cargando imágenes de test...')
df_test = pd.read_csv(TEST_CSV, sep=';')

test_images = []
for _, row in df_test.iterrows():
    img_path = GTSRB_DIR / row['Path']
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    test_images.append(np.array(img, dtype=np.float32) / 255.0)

X_test = np.array(test_images)
y_test = df_test['ClassId'].to_numpy(dtype=np.int32)
print(f'Test: {len(X_test)}')

In [ ]:
# Arquitectura CNN
def build_cnn(img_size=IMG_SIZE, num_classes=NUM_CLASSES, lr=LEARNING_RATE):
    model = models.Sequential([
        layers.Input(shape=(img_size, img_size, 3)),

        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax'),
    ], name='traffic_sign_cnn')

    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

model = build_cnn()
model.summary()

In [ ]:
# Pesos de clase para manejar el desbalance
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes.tolist(), weights.tolist()))
print(f'Pesos calculados para {len(class_weight)} clases')

In [ ]:
# Entrenamiento
cb = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(
        filepath=str(OUT_DIR / 'cnn_gtsrb_v1.h5'),
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1,
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=cb,
)
print('Entrenamiento completado')

In [ ]:
# Curvas de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validación')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Época')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validación')
axes[1].set_title('Loss')
axes[1].set_xlabel('Época')
axes[1].legend()

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'curvas_entrenamiento.png'), dpi=150)
plt.show()

In [ ]:
# Evaluación en Test
from tensorflow.keras.models import load_model

best_model = load_model(str(OUT_DIR / 'cnn_gtsrb_v1.h5'))
test_loss, test_acc = best_model.evaluate(X_test, y_test, batch_size=BATCH_SIZE, verbose=0)
print(f'Test Accuracy: {test_acc:.4f} ({test_acc:.1%})')
print(f'Test Loss:     {test_loss:.4f}')

In [ ]:
# Matriz de confusión (top 10 clases más confundidas)
from sklearn.metrics import classification_report

y_pred = np.argmax(best_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0), axis=1)
print(classification_report(y_test, y_pred))

In [ ]:
# (Opcional) Subir modelo a Hugging Face Hub directamente desde Kaggle
# Para esto debes agregar un Kaggle Secret llamado HF_TOKEN con tu token de HF

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')

    from huggingface_hub import HfApi
    api = HfApi(token=hf_token)

    repo_id = 'jarolmedina41/traffic-signs-cnn'
    api.create_repo(repo_id=repo_id, exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(OUT_DIR / 'cnn_gtsrb_v1.h5'),
        path_in_repo='cnn_gtsrb_v1.h5',
        repo_id=repo_id,
    )
    print(f'Modelo subido a https://huggingface.co/{repo_id}')
except Exception as e:
    print(f'Subida automática no disponible: {e}')
    print('Descarga manual: Output > cnn_gtsrb_v1.h5')